In [ ]:
# Fetch only observations newer than each coin's latest saved date.
from datetime import datetime, timezone
from pathlib import Path

from getpass import getpass

import pandas as pd
import requests

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
PRICE_FILE = DATA_DIR / 'crypto_daily_prices_usd.csv'
# The key is entered at runtime with hidden input and is not saved in this notebook.
api_plan = input('CoinGecko API plan (demo/pro): ').strip().lower()
if api_plan not in {'demo', 'pro'}:
    raise ValueError("Enter 'demo' or 'pro' for your CoinGecko API plan.")
api_key = getpass('CoinGecko API key (input hidden): ').strip()
if not api_key:
    raise ValueError('A CoinGecko API key is required.')
api_host = 'api.coingecko.com' if api_plan == 'demo' else 'pro-api.coingecko.com'
api_header = 'x-cg-demo-api-key' if api_plan == 'demo' else 'x-cg-pro-api-key'
headers = {api_header: api_key}
COIN_IDS = {
    'BTC': 'bitcoin',
    'BAT': 'basic-attention-token',
    'VVV': 'venice-token',
    'NEAR': 'near',
    'ZEC': 'zcash',
    'GRT': 'the-graph',
    'AAVE': 'aave',
    'AERO': 'aerodrome-finance',
    'HYPE': 'hyperliquid',
}

END_DATE = datetime.now(timezone.utc)
prices = pd.read_csv(PRICE_FILE, parse_dates=['date'])
prices['date'] = pd.to_datetime(prices['date'], utc=True).dt.normalize()
prices = prices.set_index('date').sort_index()

for ticker, coin_id in COIN_IDS.items():
    if ticker not in prices.columns:
        prices[ticker] = pd.NA
    saved = pd.to_numeric(prices[ticker], errors='coerce').dropna()
    start_date = (saved.index.max() - pd.Timedelta(days=1)).to_pydatetime() if not saved.empty else datetime(2025, 10, 1, tzinfo=timezone.utc)
    start_date = start_date.replace(hour=0, minute=0, second=0, microsecond=0)
    url = f'https://{api_host}/api/v3/coins/{coin_id}/market_chart/range'
    params = {'vs_currency': 'usd', 'from': int(start_date.timestamp()), 'to': int(END_DATE.timestamp()), 'interval': 'daily'}
    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()
    data = response.json()
    if 'prices' not in data:
        raise ValueError(f'{ticker}: CoinGecko response did not contain prices: {data}')
    fetched = pd.DataFrame(data['prices'], columns=['timestamp_ms', ticker])
    if fetched.empty:
        latest = saved.index.max().date() if not saved.empty else 'none'
        print(f'{ticker}: no newer observations (latest saved: {latest})')
        continue
    fetched['date'] = pd.to_datetime(fetched.pop('timestamp_ms'), unit='ms', utc=True).dt.normalize()
    updates = fetched.drop_duplicates('date', keep='last').set_index('date')[ticker]
    prices = prices.reindex(prices.index.union(updates.index)).sort_index()
    prices.loc[updates.index, ticker] = updates
    print(f'{ticker}: saved {len(updates)} observations from {updates.index.min().date()} through {updates.index.max().date()}')

prices = prices.sort_index()
prices.to_csv(PRICE_FILE, date_format='%Y-%m-%dT%H:%M:%SZ')
print(f'Updated {PRICE_FILE} for all requested coins.')